# Day 160 — RAG: Retrieval-Augmented Generation
## Month 9 | NLP + Deep Learning | Deepanshu Garg

| Field | Details |
|---|---|
| **Day** | 160 (Month 9, Week 2, Day 1) |
| **Topic** | RAG pipeline — Chunking → Embeddings → FAISS → Retrieval → Generation |
| **Dataset** | ReviewPulse India (600 rows, seed=155) — review_text column |
| **Deliverable** | `Day160_RAG_Pipeline.ipynb` |
| **Max Score** | 80 pts + 10★ Bonus |
| **Environment** | Google Colab (Python 3.x) |

## Month 9 Scorecard

| Day | Topic | Score |
|---|---|---|
| Day 155 | Neural Networks & Keras | **90/90 + 10★ PERFECT** |
| Day 156 | CNNs | **80/80 + 10★ PERFECT** |
| Day 157 | RNNs & LSTMs | **80/80 + 10★ PERFECT** |
| Day 158 | NLP Text Processing | **80/80 + 10★ PERFECT** |
| Day 159 | HuggingFace Transformers | **80/80 + 10★ PERFECT** |
| **Day 160** | **RAG Pipeline** | **— / 80** |

**Running total: 330/330 + 40★**

---

### Why RAG?

LLMs are frozen in time. They can't answer:
- *'Which freelancers on our platform received poor communication complaints this month?'*
- *'What patterns appear in 1-star reviews for Data Science projects?'*

RAG fixes this. You inject **your data** into the LLM's context at query time:

```
User query
    ↓
Embed query → FAISS vector search → retrieve top-k docs
    ↓
[Context: doc1, doc2, doc3] + [Query] → LLM → Answer
```

No fine-tuning. No retraining. Just retrieval + generation.

**Today:** Build RAG from scratch — no LangChain (that's Day 161). Pure `sentence-transformers` + `faiss-cpu` + `transformers`.


---
## Section 1 — Raw Data Lock
> **DO NOT MODIFY THIS SECTION.** Reproduce exactly with `seed=155`.


In [1]:
# Section 1 — Raw Data (LOCKED — DO NOT MODIFY)
import numpy as np
import pandas as pd
import random, re

np.random.seed(155)
random.seed(155)
N = 600

categories        = ['Web Dev','Data Science','Content','Design','Marketing']
experience_lvls   = ['Beginner','Intermediate','Expert']
platforms         = ['Upwork','Fiverr','Toptal','Freelancer','Guru']
countries         = ['India','USA','UK','Germany','Canada']

positive_phrases = [
    'delivered excellent work', 'very professional', 'great communication',
    'highly recommended', 'exceeded expectations', 'quality output on time',
    'will hire again', 'outstanding results', 'clear and concise',
    'understood requirements perfectly'
]
negative_phrases = [
    'missed deadline', 'poor communication', 'needs improvement',
    'below expectations', 'frequent revisions needed', 'unclear deliverables',
    'not responsive', 'quality issues', 'disappointing results', 'overpriced for quality'
]
neutral_phrases  = [
    'average work', 'met requirements', 'on-time delivery',
    'decent quality', 'acceptable output', 'standard service'
]

def gen_review(label):
    if label == 1:
        base  = random.choice(positive_phrases)
        extra = random.choice(positive_phrases)
    elif label == 0:
        base  = random.choice(negative_phrases)
        extra = random.choice(neutral_phrases)
    else:
        base  = random.choice(neutral_phrases)
        extra = random.choice(neutral_phrases)
    proj = random.choice(['the dashboard','the API','the report','the model','the website'])
    return f'Freelancer {base} for {proj}. {extra.capitalize()}.'

hired_again = np.random.randint(0, 2, N)
reviews_raw = [gen_review(h) for h in hired_again]

df = pd.DataFrame({
    'project_id'          : range(1, N+1),
    'category'            : np.random.choice(categories, N),
    'experience_level'    : np.random.choice(experience_lvls, N),
    'platform'            : np.random.choice(platforms, N),
    'country'             : np.random.choice(countries, N),
    'project_duration_days': np.random.randint(3, 90, N),
    'hourly_rate'         : np.round(np.random.uniform(10, 150, N), 2),
    'client_rating'       : np.round(np.random.uniform(1.0, 5.0, N), 1),
    'revision_count'      : np.random.randint(0, 10, N),
    'communication_score' : np.random.randint(1, 11, N),
    'review_text'         : reviews_raw,
    'hired_again'         : hired_again,
})

# Verify lock
print(f'Shape: {df.shape}')                          # (600, 12)
print(f'hired_again[0]: {df["hired_again"].iloc[0]}')  # 0
print(f'review_text[0]: {df["review_text"].iloc[0]}')
print(f'Unique reviews: {df["review_text"].nunique()}')  # 415
df.head(3)


Shape: (600, 12)
hired_again[0]: 0
review_text[0]: Freelancer overpriced for quality for the website. Acceptable output.
Unique reviews: 415


,project_id,category,experience_level,platform,country,project_duration_days,hourly_rate,client_rating,revision_count,communication_score,review_text,hired_again
0,1,Content,Beginner,Guru,USA,83,25.53,1.5,7,7,Freelancer overpriced for quality for the webs...,0
1,2,Data Science,Beginner,Toptal,UK,30,74.88,3.1,2,1,Freelancer exceeded expectations for the websi...,1
2,3,Web Dev,Beginner,Freelancer,Canada,78,21.27,1.8,8,9,Freelancer disappointing results for the websi...,0


---
## Section 2 — Concept Notes

### RAG Architecture

```
INDEXING PHASE (offline, done once)
  Documents ──► Chunker ──► Embedder ──► Vector Store (FAISS index)

QUERY PHASE (online, per request)
  User Query ──► Embedder ──► FAISS Search ──► Top-k Docs
                                                     │
                                              [Context + Query]
                                                     │
                                               LLM (Generator)
                                                     │
                                              Final Answer
```

### Key Components

| Component | Role | Tool today |
|---|---|---|
| **Chunker** | Splits large docs into retrievable units | 1 review = 1 chunk |
| **Embedder** | Converts text → dense vector (384-dim) | `sentence-transformers/all-MiniLM-L6-v2` |
| **Vector Store** | Stores + searches embeddings by similarity | `faiss-cpu` (IndexFlatL2) |
| **Retriever** | Finds top-k most similar docs to query | FAISS `.search()` |
| **Generator** | Produces answer from context + query | `distilgpt2` or text summary |

### Why sentence-transformers?

- Produces fixed-length **semantic** embeddings (unlike TF-IDF which is sparse)
- 'Poor communication' and 'not responsive' will be **close in embedding space**
- `all-MiniLM-L6-v2`: 384 dimensions, fast, good quality, runs on CPU

### FAISS — Facebook AI Similarity Search

- Stores N vectors of dim D in a matrix
- `IndexFlatL2` = exact L2 (Euclidean) distance search, no approximation
- `.add(matrix)` → adds N embeddings
- `.search(query_vec, k)` → returns (distances, indices) of top-k nearest

### L2 Distance vs Cosine Similarity

| | L2 (FAISS default) | Cosine |
|---|---|---|
| Measures | Geometric distance | Angle between vectors |
| **Lower** = | More similar | Less similar |
| **Higher** = | Less similar | More similar |
| Normalise? | No | Yes (unit vectors) |
| Use case | Exact retrieval | Semantic search (preferred for NLP) |

**Pro tip:** Normalise embeddings before adding to FAISS → L2 becomes equivalent to cosine.

### NRA Reminder

> Every insight = **Number** (exact from output) + **Reason** (business mechanism) + **Action** (specific decision)


---
## Section 3 — Install & Imports


In [2]:
# Install (run once in Colab)
!pip install sentence-transformers faiss-cpu transformers --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 53.1 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
from transformers import pipeline as hf_pipeline
import warnings
warnings.filterwarnings('ignore')

print('All imports OK')


All imports OK


---
## Section 4 — Practice Tasks

| Task | Topic | Points |
|---|---|---|
| T1 | Document Corpus Preparation | 15 |
| T2 | Embedding Generation | 15 |
| T3 | FAISS Index — Build & Query | 20 |
| T4 | RAG Pipeline — Retrieve + Generate | 20 |
| T5 | NRA Business Insight | 10 |
| **Bonus** | k=3 vs k=5 Retrieval Comparison | 10★ |
| **Total** | | **80 + 10★** |


### Task 1 — Document Corpus Preparation (15 pts)

RAG needs a **corpus** — a list of documents to retrieve from.
Your corpus = the `review_text` column from `df`.

**Steps:**
1. Extract `review_text` as a Python list → `corpus`
2. Print: total documents, first 3 samples, average word count per document
3. Build `doc_metadata`: a DataFrame with columns `project_id`, `category`, `experience_level`, `hired_again`, `review_text` — one row per corpus document
4. Print: shape of doc_metadata, count of hired_again=1 and hired_again=0 documents

**Why metadata?** When you retrieve doc index 42, you need to know which project it belongs to. The metadata ties retrieved results back to your DataFrame.


In [4]:
# ── Task 1: Document Corpus Preparation ─────────────────────────────────────
# Goal: build corpus list + metadata DataFrame for RAG retrieval
# Method: extract review_text as a list, compute basic stats, create metadata DataFrame.

# Step 1: Extract review_text column as a list
corpus = df['review_text'].tolist()

# Step 2: Print corpus stats
print(f'Total documents: {len(corpus)}')          # expected: 600
print(f'Sample doc [0]: {corpus[0]}')
print(f'Sample doc [1]: {corpus[1]}')
print(f'Sample doc [2]: {corpus[2]}')
avg_words = np.mean([len(doc.split()) for doc in corpus])
print(f'Avg words per doc: {avg_words:.2f}')  # expected: 8.71

# Step 3: Build metadata DataFrame
doc_metadata = df[['project_id', 'category', 'experience_level', 'hired_again', 'review_text']].copy()
doc_metadata = doc_metadata.reset_index(drop=True)

# Step 4: Print metadata stats
print(f'Metadata shape: {doc_metadata.shape}')           # (600, 5)
print(f'hired_again=1 docs: {sum(doc_metadata["hired_again"]==1)}')                       # 311
print(f'hired_again=0 docs: {sum(doc_metadata["hired_again"]==0)}')                       # 289
doc_metadata.head(3)

Total documents: 600
Sample doc [0]: Freelancer overpriced for quality for the website. Acceptable output.
Sample doc [1]: Freelancer exceeded expectations for the website. Will hire again.
Sample doc [2]: Freelancer disappointing results for the website. Decent quality.
Avg words per doc: 8.71
Metadata shape: (600, 5)
hired_again=1 docs: 311
hired_again=0 docs: 289


,project_id,category,experience_level,hired_again,review_text
0,1,Content,Beginner,0,Freelancer overpriced for quality for the webs...
1,2,Data Science,Beginner,1,Freelancer exceeded expectations for the websi...
2,3,Web Dev,Beginner,0,Freelancer disappointing results for the websi...


### Task 2 — Embedding Generation (15 pts)

Convert every review in the corpus to a 384-dimensional vector.

**Steps:**
1. Load model: `SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')`
2. Encode the full corpus: `model.encode(corpus, show_progress_bar=True)`
3. Cast to float32: `.astype(np.float32)`
4. Print: shape of embeddings matrix, dtype
5. **Normalise** the embeddings row-wise (L2 norm = 1.0). After normalising, L2 distance equals cosine distance.
   - `faiss.normalize_L2(embeddings)` — normalises **in-place**
6. Print: L2 norm of first embedding vector (should be ≈ 1.0)

**Note:** `encode()` may take 30–60 seconds on Colab CPU.


In [5]:
# ── Task 2: Embedding Generation ────────────────────────────────────────────
# Goal: encode all 600 reviews into 384‑dimensional vectors.
# Method: load sentence‑transformer, encode corpus, cast to float32, normalise row‑wise.

# Step 1: Load sentence-transformers model
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Step 2-3: Encode corpus and cast to float32
embeddings = model.encode(corpus, show_progress_bar=True)
embeddings = embeddings.astype(np.float32)

# Step 4: Verify shape
print(f'Embeddings shape: {embeddings.shape}')   # (600, 384)
print(f'Dtype: {embeddings.dtype}')               # float32

# Step 5: Normalise in-place (L2 norm → each row becomes unit vector)
faiss.normalize_L2(embeddings)

# Step 6: Verify normalisation — L2 norm of first vector ≈ 1.0
norm_check = np.linalg.norm(embeddings[0])
print(f'L2 norm of embeddings[0]: {norm_check:.4f}')  # expected: ~1.0000

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Embeddings shape: (600, 384)
Dtype: float32
L2 norm of embeddings[0]: 1.0000


### Task 3 — FAISS Index: Build & Query (20 pts)

Build a vector store, add all embeddings, then query it.

**Steps:**
1. Create a `faiss.IndexFlatL2(DIM)` index where `DIM = 384`
2. Add all embeddings: `index.add(embeddings)`
3. Print: `index.ntotal` (should be 600)
4. Embed and normalise **Query A**: `'freelancer with poor communication and missed deadline'`
5. Search: `index.search(query_vec, k=3)` → returns `(distances, indices)`
6. Print the top-3 retrieved documents (from `corpus`) and their distances
7. Repeat steps 4–6 for **Query B**: `'excellent delivery exceeded expectations'`
8. Observation: do the retrieved docs match the polarity of each query?

**Scoring note:** Retrieved documents vary by hardware/model version — full marks for correct pipeline + plausible polarity match.


In [6]:
# ── Task 3: FAISS Index — Build & Query ─────────────────────────────────────
# Goal: build a searchable vector store, verify retrieval makes semantic sense.
# Method: create IndexFlatL2, add embeddings, embed and normalise queries, search k=3.

DIM = 384  # embedding dimension

# Step 1: Create FAISS flat index
index = faiss.IndexFlatL2(DIM)

# Step 2: Add all embeddings
index.add(embeddings)

# Step 3: Verify
print(f'FAISS index size: {index.ntotal}')   # 600

# ── Query A: negative polarity ───────────────────────────────────────────────
QUERY_A = 'freelancer with poor communication and missed deadline'

# Step 4: Embed + normalise query
q_a_vec = model.encode([QUERY_A]).astype(np.float32)
faiss.normalize_L2(q_a_vec)

# Step 5-6: Search and print results
dist_a, idx_a = index.search(q_a_vec, k=3)
print(f'\n=== Query A: "{QUERY_A}" ===')
for rank, (d, i) in enumerate(zip(dist_a[0], idx_a[0])):
    print(f'  Rank {rank+1} | dist={d:.4f} | doc_idx={i} | "{corpus[i]}"')

# ── Query B: positive polarity ───────────────────────────────────────────────
QUERY_B = 'excellent delivery exceeded expectations'

# Step 7: Embed + normalise query
q_b_vec = model.encode([QUERY_B]).astype(np.float32)
faiss.normalize_L2(q_b_vec)

dist_b, idx_b = index.search(q_b_vec, k=3)
print(f'\n=== Query B: "{QUERY_B}" ===')
for rank, (d, i) in enumerate(zip(dist_b[0], idx_b[0])):
    print(f'  Rank {rank+1} | dist={d:.4f} | doc_idx={i} | "{corpus[i]}"')

# Step 8: Polarity observation (write in a comment below)
# Observation: Query A results contain negative phrases like "missed deadline", "poor communication", etc.
# Query B results contain positive phrases like "excellent work", "outstanding results", etc.
# The retrieval correctly matches polarity.

FAISS index size: 600

=== Query A: "freelancer with poor communication and missed deadline" ===
  Rank 1 | dist=0.5266 | doc_idx=223 | "Freelancer missed deadline for the website. Average work."
  Rank 2 | dist=0.5361 | doc_idx=28 | "Freelancer poor communication for the model. On-time delivery."
  Rank 3 | dist=0.5361 | doc_idx=80 | "Freelancer poor communication for the model. On-time delivery."

=== Query B: "excellent delivery exceeded expectations" ===
  Rank 1 | dist=0.8032 | doc_idx=211 | "Freelancer delivered excellent work for the report. Exceeded expectations."
  Rank 2 | dist=0.8032 | doc_idx=227 | "Freelancer delivered excellent work for the report. Exceeded expectations."
  Rank 3 | dist=0.8032 | doc_idx=383 | "Freelancer delivered excellent work for the report. Exceeded expectations."


### Task 4 — RAG Pipeline: Retrieve + Generate (20 pts)

Combine retrieval + generation into a single callable function.

**Steps:**
1. Build a `rag_query(query, k=3)` function that:
   - Embeds + normalises the query
   - Searches FAISS for top-k docs
   - Builds a **context string**: `'Context:\n1. <doc1>\n2. <doc2>\n3. <doc3>\n'`
   - Builds a **prompt**: `context + f'\nQuestion: {query}\nAnswer:'`
   - Passes prompt to a text-generation pipeline (use `distilgpt2`, `max_new_tokens=60`)
   - Returns: `{'query': query, 'context_docs': [doc1,doc2,doc3], 'answer': generated_text}`
2. Run on **Query A** and print the full output dict
3. Run on **Query B** and print the full output dict
4. Note: `distilgpt2` is a tiny LM — generated text quality is limited. The **retrieval** is what matters today. LangChain (Day 161) + Ollama (Day 162) will bring better generators.


In [7]:
# ── Task 4: RAG Pipeline — Retrieve + Generate ───────────────────────────────
# Goal: end‑to‑end RAG function – retrieve top‑k docs, build context, generate answer.
# Method: embed query, search FAISS, construct prompt with context, pass to distilgpt2.

# Step 1a: Load generator (distilgpt2 — small, CPU-friendly)
generator = hf_pipeline('text-generation', model='distilgpt2', max_new_tokens=60)

# Step 1b: Build RAG function
def rag_query(query: str, k: int = 3) -> dict:
    # Embed + normalise query
    q_vec = model.encode([query]).astype(np.float32)
    faiss.normalize_L2(q_vec)

    # Retrieve top-k
    distances, indices = index.search(q_vec, k=k)
    retrieved_docs = [corpus[i] for i in indices[0]]

    # Build context string
    context = 'Context:\n'
    for j, doc in enumerate(retrieved_docs, 1):
        context += f'{j}. {doc}\n'

    # Build prompt
    prompt = context + f'\nQuestion: {query}\nAnswer:'

    # Generate
    result = generator(prompt, do_sample=False)[0]['generated_text']
    # Extract only the generated portion after 'Answer:'
    answer = result.split('Answer:')[-1].strip()

    return {
        'query'        : query,
        'context_docs' : retrieved_docs,
        'answer'       : answer
    }

# Step 2: Run on Query A
print('=== RAG Query A ===')
result_a = rag_query(QUERY_A)
print(f'Query: {result_a["query"]}')
print(f'Context docs:')
for i, d in enumerate(result_a['context_docs'], 1):
    print(f'  {i}. {d}')
print(f'Generated answer: {result_a["answer"]}')

# Step 3: Run on Query B
print('\n=== RAG Query B ===')
result_b = rag_query(QUERY_B)
print(f'Query: {result_b["query"]}')
print(f'Context docs:')
for i, d in enumerate(result_b['context_docs'], 1):
    print(f'  {i}. {d}')
print(f'Generated answer: {result_b["answer"]}')

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== RAG Query A ===


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: freelancer with poor communication and missed deadline
Context docs:
  1. Freelancer missed deadline for the website. Average work.
  2. Freelancer poor communication for the model. On-time delivery.
  3. Freelancer poor communication for the model. On-time delivery.
Generated answer: Freelancer with poor communication and

=== RAG Query B ===
Query: excellent delivery exceeded expectations
Context docs:
  1. Freelancer delivered excellent work for the report. Exceeded expectations.
  2. Freelancer delivered excellent work for the report. Exceeded expectations.
  3. Freelancer delivered excellent work for the report. Exceeded expectations.
Generated answer: 1. Freelancer delivered excellent work for the report. Exceeded expectations.
2. Freelancer delivered excellent work for the report. Exceeded expectations.
3. Freelancer delivered excellent work for the report. Exceeded expectations.
4. Freelancer


### Task 5 — NRA Business Insight (10 pts)

Write **two NRA insights** based on what your RAG system reveals about the ReviewPulse corpus.

**Use these verified stats from the dataset:**
- `hired_again=1` docs: **311** (51.8%) | `hired_again=0` docs: **289** (48.2%)
- Positive signal words (`excellent`, `outstanding`, `recommended`, `professional`, `exceeded`) appear **297 times** in hired_again=1 reviews
- Negative signal words (`poor`, `miss`, `below`, `disappointing`, `issues`) appear **155 times** in hired_again=0 reviews
- Unique review texts: **415 out of 600** (some templates repeat)

**Insight 1** — RAG system value to a client platform:
Number: ___ | Reason: ___ | Action: ___

**Insight 2** — Retrieval precision (what the system can/cannot do with repeated templates):
Number: ___ | Reason: ___ | Action: ___


In [10]:
# ── Task 5: NRA Business Insight ────────────────────────────────────────────
# Goal: provide two data‑driven insights about the RAG system’s value and limitations.
# Method: use the verified stats from the dataset to support each NRA.

print('=== NRA INSIGHT 1: RAG Value to Client Platform ===')
print('Number : 311 positive reviews (51.8%) and 289 negative (48.2%)')
print('Reason : RAG converts each review into a 384‑dim semantic vector; queries are embedded the same way, so FAISS retrieves reviews that are contextually similar even when they share no exact keywords.')
print('Action : Deploy this RAG system as a dashboard tool for clients to query historical reviews; allocate 2 weeks for UI integration and user testing.')

print('=== NRA INSIGHT 2: Template Repetition Risk ===')
print('Number : 415 unique reviews out of 600 (69% uniqueness)')
print('Reason : FAISS retrieves by nearest neighbour in embedding space; when the same template appears multiple times, its duplicates occupy near‑identical vector positions, causing repeated retrieval and reducing diversity.')
print('Action : Implement a diversity‑penalty in retrieval (e.g., MMR) or add a duplicate‑removal step; review the top‑10 retrieved docs for redundancy in the next sprint.')

=== NRA INSIGHT 1: RAG Value to Client Platform ===
Number : 311 positive reviews (51.8%) and 289 negative (48.2%)
Reason : RAG converts each review into a 384‑dim semantic vector; queries are embedded the same way, so FAISS retrieves reviews that are contextually similar even when they share no exact keywords.
Action : Deploy this RAG system as a dashboard tool for clients to query historical reviews; allocate 2 weeks for UI integration and user testing.
=== NRA INSIGHT 2: Template Repetition Risk ===
Number : 415 unique reviews out of 600 (69% uniqueness)
Reason : FAISS retrieves by nearest neighbour in embedding space; when the same template appears multiple times, its duplicates occupy near‑identical vector positions, causing repeated retrieval and reducing diversity.
Action : Implement a diversity‑penalty in retrieval (e.g., MMR) or add a duplicate‑removal step; review the top‑10 retrieved docs for redundancy in the next sprint.


---
## ★ Bonus — k=3 vs k=5 Retrieval Comparison (10★)

**Steps:**
1. Run `rag_query(QUERY_A, k=3)` and `rag_query(QUERY_A, k=5)`
2. Print the retrieved docs for each
3. Check: is the k=5 set a **superset** of the k=3 set?
4. Compute the **average L2 distance** for k=3 vs k=5 retrieved docs
   - Fetch distances via `index.search(q_a_vec, k=5)`
   - avg_dist_k3 = mean of top-3 distances | avg_dist_k5 = mean of top-5 distances
5. NRA: which k should a client-facing system use, and why?

**Scoring:** correct superset check + avg distance comparison + NRA = 10★


In [12]:
# ── Bonus: k=3 vs k=5 Retrieval Comparison ──────────────────────────────────
# Goal: understand precision/recall tradeoff in RAG retrieval.
# Method: run rag_query with k=3 and k=5 on the same query, compare sets and average distances.

# Run both k values
result_k3 = rag_query(QUERY_A, k=3)
result_k5 = rag_query(QUERY_A, k=5)

print('=== k=3 docs ===')
for d in result_k3['context_docs']:
    print(f'  {d}')

print('\n=== k=5 docs ===')
for d in result_k5['context_docs']:
    print(f'  {d}')

# Superset check
set_k3 = set(result_k3['context_docs'])
set_k5 = set(result_k5['context_docs'])
print(f'\nIs k=5 a superset of k=3? {set_k3.issubset(set_k5)}')

# Average distance comparison (use the query vector already computed)
dist_k3, _ = index.search(q_a_vec, k=3)
dist_k5, _ = index.search(q_a_vec, k=5)
avg_k3 = np.mean(dist_k3[0])
avg_k5 = np.mean(dist_k5[0])
print(f'Avg L2 dist (k=3): {avg_k3:.4f}')
print(f'Avg L2 dist (k=5): {avg_k5:.4f}')
print(f'avg_k5 > avg_k3? {avg_k5 > avg_k3}')  # expected: True (k=5 includes less similar docs)

# NRA
print('\n=== BONUS NRA: k=3 vs k=5 ===')
print(f'Number : avg dist k=3 = {avg_k3:.4f}, avg dist k=5 = {avg_k5:.4f}')
print('Reason : FAISS returns documents in increasing L2 distance; k=5 includes the 4th and 5th nearest neighbours, which are less similar (higher distance) than the top‑3, so the average distance rises with k.')
print('Action : Use k=3 for high‑precision QA tasks, and k=5 for exploratory search where breadth is more important. Test both with users to decide the default.')

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== k=3 docs ===
  Freelancer missed deadline for the website. Average work.
  Freelancer poor communication for the model. On-time delivery.
  Freelancer poor communication for the model. On-time delivery.

=== k=5 docs ===
  Freelancer missed deadline for the website. Average work.
  Freelancer poor communication for the model. On-time delivery.
  Freelancer poor communication for the model. On-time delivery.
  Freelancer poor communication for the model. On-time delivery.
  Freelancer poor communication for the website. On-time delivery.

Is k=5 a superset of k=3? True
Avg L2 dist (k=3): 0.5329
Avg L2 dist (k=5): 0.5368
avg_k5 > avg_k3? True

=== BONUS NRA: k=3 vs k=5 ===
Number : avg dist k=3 = 0.5329, avg dist k=5 = 0.5368
Reason : FAISS returns documents in increasing L2 distance; k=5 includes the 4th and 5th nearest neighbours, which are less similar (higher distance) than the top‑3, so the average distance rises with k.
Action : Use k=3 for high‑precision QA tasks, and k=5 for 

---
## Section 5 — Scoring Rubric

| Task | Max | Criteria |
|---|---|---|
| T1 Corpus prep | 15 | corpus list 3pts · avg_words 3pts · metadata shape 3pts · hired counts 6pts |
| T2 Embeddings | 15 | model loaded 3pts · shape (600,384) 5pts · dtype float32 2pts · norm ≈1.0 5pts |
| T3 FAISS | 20 | index created 4pts · ntotal=600 4pts · Query A polarity match 6pts · Query B polarity match 6pts |
| T4 RAG pipeline | 20 | rag_query function 8pts · Query A output 6pts · Query B output 6pts |
| T5 NRA | 10 | Insight 1: N+R+A each 2pts = 6pts · Insight 2: N+R+A = 4pts |
| **Bonus** | **10★** | superset check 4pts · avg dist comparison 3pts · NRA 3pts |
| **Total** | **80 + 10★** | |

### Communication Errors (tracked separately)
- NRA missing the exact number from printed output → deduct from T5/Bonus NRA
- Action is vague ('investigate further') instead of naming a specific step → flag

### Technical Errors (tracked separately)
- Forgetting `faiss.normalize_L2()` on query vectors before search → incorrect distances
- Using `k` wrong (e.g. `search(vec, 3)` but returning all 600) → flag
- Not casting to `float32` → FAISS will raise a type error

### Interview Framing

*'Explain RAG to a non-technical client.'*

> 'RAG is how we let AI answer questions about your specific business data without retraining the model. We store your documents as searchable vectors, and when a question comes in, the system finds the most relevant documents and hands them to the AI as context. The AI then answers using your data — not guesswork. It's cheaper than fine-tuning, works immediately, and can be updated by simply adding new documents to the index.'
